# AMR Cascade Platform Debug Notebook

Purpose: inspect the current pipeline inputs, intermediate artifacts, model inputs, and final outputs before running production on HPC.

This notebook is intentionally audit-oriented. It should answer four questions:

1. What data layers exist right now?
2. Which pairs enter cascade estimation and validation?
3. Does adjusted downstream-testing regression use only validated `robust` / `supported` pairs?
4. Does prevalence-shift analysis run only for downstream antibiotics from validated escalation edges?

Default behavior is read-only. It does not regenerate artifacts unless you explicitly set `RUN_OPTIONAL_STAGES = True`.


## 0. Setup

Use the same config/context loader as the CLI. If this fails, the project environment is not ready and HPC should not be started.


In [ ]:
import sys
from pathlib import Path
import json
import math

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "amr_cascade_platform").exists():
    PROJECT_ROOT = Path("/Users/awotoroebenezer/Desktop/amr_cascade_platform")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from amr_cascade_platform.cli.main import build_context
from amr_cascade_platform.core.utils.antibiotic_names import normalize_antibiotic_label

settings, path_manager, mapper = build_context(PROJECT_ROOT, "mac")

ORGANISM_LABEL = "ESCHERICHIA COLI"
ORGANISM_SLUG = "escherichia_coli"
SCOPE = "combined"
SITE = None
RUN_OPTIONAL_STAGES = False

print("project:", PROJECT_ROOT)
print("environment:", settings.environment.name)
print("data root:", path_manager.paths.raw.parent)
print("organism label:", ORGANISM_LABEL)
print("organism slug:", ORGANISM_SLUG)


## 1. Current Scientific Contract

The corrected production contract is:

- **Cascade estimation** starts with episode-pair rows.
- **Retention** uses raw observation evidence only: support, informative downstream testing, panel-bundling status, and raw ER.
- **Validation** assigns `robust`, `supported`, `mixed`, or `insufficient`.
- **Adjusted downstream-testing regression** is fitted **after validation** and only for `robust` / `supported` pairs.
- **Prevalence-shift analysis** is restricted to downstream antibiotics from validated **escalation** edges.
- Suppression edges can be valid cascade findings, but they are not used as resistance-triggered prevalence anchors.


In [ ]:
from amr_cascade_platform.cascade.workflows.cascade_analysis_workflow import CascadeAnalysisWorkflow

print("Adjusted model helper:", CascadeAnalysisWorkflow._validated_edge_results.__name__)
print(CascadeAnalysisWorkflow._validated_edge_results.__doc__)


## 2. Artifact Map

This cell lists the specific files the notebook expects. Missing files are not automatically an error; they mean that stage has not been run yet or was deleted before a fresh run.


In [ ]:
def exists_table(path: Path) -> dict:
    return {"path": str(path), "exists": path.exists(), "size_mb": round(path.stat().st_size / 1e6, 2) if path.exists() else None}

gold_dir = path_manager.paths.gold / SCOPE / "organisms" / ORGANISM_SLUG
cascade_dir = path_manager.paths.artifacts / settings.cascade.outputs.result_dir / SCOPE / "organisms" / ORGANISM_SLUG
prevalence_dir = path_manager.paths.artifacts / settings.prevalence.output_dir / SCOPE / "organisms" / ORGANISM_SLUG
report_dir = path_manager.paths.outputs / "reports" / SCOPE / "organisms" / ORGANISM_SLUG

expected = {
    "gold_culture_episodes": gold_dir / "culture_episodes.parquet",
    "gold_culture_drug_episodes": gold_dir / "culture_drug_episodes.parquet",
    "gold_eligible_pairs": gold_dir / "eligible_pairs.parquet",
    "gold_drug_pair_episodes": gold_dir / "drug_pair_episodes.parquet",
    "cascade_edge_report": cascade_dir / "edge_report.parquet",
    "cascade_validation_results": cascade_dir / "validation_results.parquet",
    "cascade_adjusted_diagnostics": cascade_dir / "adjusted_model_diagnostics.parquet",
    "prevalence_shift": prevalence_dir / "prevalence_shift.parquet",
        "prevalence_legacy_delta_curve": prevalence_dir / "legacy" / "legacy_delta_sensitivity_curves.parquet",
    "prevalence_mnar_curve": prevalence_dir / "prevalence_mnar_sensitivity_curves.parquet",
    "report_manifest": report_dir / "report_manifest.json",
}

pd.DataFrame([{"artifact": k, **exists_table(v)} for k, v in expected.items()])


## 3. Gold Layer Inspection

This verifies the denominator objects before any model is considered:

- `culture_episodes`: organism-level analysis units.
- `culture_drug_episodes`: directly observed AST rows.
- `eligible_pairs`: episode-drug opportunity denominator.
- `drug_pair_episodes`: directional upstream-downstream pair rows.


In [ ]:
def read_parquet_if_exists(path: Path, columns=None) -> pd.DataFrame:
    if not path.exists():
        print("MISSING:", path)
        return pd.DataFrame()
    return pd.read_parquet(path, columns=columns)

culture_episodes = read_parquet_if_exists(expected["gold_culture_episodes"])
culture_drug_episodes = read_parquet_if_exists(expected["gold_culture_drug_episodes"])
eligible_pairs = read_parquet_if_exists(expected["gold_eligible_pairs"])
drug_pairs = read_parquet_if_exists(expected["gold_drug_pair_episodes"])

rows = []
for name, df in {
    "culture_episodes": culture_episodes,
    "culture_drug_episodes": culture_drug_episodes,
    "eligible_pairs": eligible_pairs,
    "drug_pair_episodes": drug_pairs,
}.items():
    rows.append({"table": name, "rows": len(df), "columns": len(df.columns), "empty": df.empty})

pd.DataFrame(rows)


In [ ]:
# Show schemas compactly.
for name, df in {
    "culture_episodes": culture_episodes,
    "culture_drug_episodes": culture_drug_episodes,
    "eligible_pairs": eligible_pairs,
    "drug_pair_episodes": drug_pairs,
}.items():
    print("\n==", name, "==")
    if df.empty:
        print("empty or missing")
    else:
        print(list(df.columns))
        display(df.head(3))


## 4. Denominator and Observation-State Audit

This checks whether the denominator is behaving as intended:

- `is_eligible == 1` defines the eligible opportunity space.
- Observed AST rows should be a subset of eligible opportunities after matching on the episode key plus antibiotic.
- Non-binary observed results are not binary-evaluable prevalence outcomes.


In [ ]:
join_keys = list(settings.gold.episode_key_columns)
pair_keys = join_keys + ["antibiotic"]

if not eligible_pairs.empty:
    denom_summary = eligible_pairs.assign(
        antibiotic_norm=eligible_pairs["antibiotic"].map(normalize_antibiotic_label)
    ).groupby("is_eligible", dropna=False).size().reset_index(name="rows")
    display(denom_summary)

if not culture_drug_episodes.empty:
    observed_summary = (
        culture_drug_episodes.assign(antibiotic_norm=culture_drug_episodes["antibiotic"].map(normalize_antibiotic_label))
        .groupby("susceptibility", dropna=False)
        .size()
        .reset_index(name="observed_rows")
        .sort_values("observed_rows", ascending=False)
    )
    display(observed_summary.head(20))


## 5. Episode-Pair Unit-of-Analysis Check

The pair table must have one row per configured episode/upstream/downstream pair. Duplicate rows inflate support, branch probabilities, validation, and prevalence triggers.


In [ ]:
if not drug_pairs.empty:
    pair_unit_keys = join_keys + ["upstream_antibiotic", "downstream_antibiotic"]
    missing = [c for c in pair_unit_keys if c not in drug_pairs.columns]
    if missing:
        print("Cannot check uniqueness; missing columns:", missing)
    else:
        duplicate_n = int(drug_pairs.duplicated(pair_unit_keys).sum())
        print("duplicate episode/upstream/downstream rows:", duplicate_n)
        if duplicate_n:
            display(drug_pairs.loc[drug_pairs.duplicated(pair_unit_keys, keep=False), pair_unit_keys + ["upstream_susceptibility", "downstream_tested"]].head(20))
        else:
            print("OK: one row per episode/upstream/downstream pair.")


## 6. Cascade Estimation: Pre-Validation Inputs

This reproduces the cascade estimation steps without running the slow validation loop:

1. Panel-bundling / co-testing filter.
2. Conditional probabilities.
3. Escalation ratio.
4. Raw retention before validation.

Adjusted regression is deliberately **not** run here.


In [ ]:
from amr_cascade_platform.cascade.analyzers.cotesting_filter_analyzer import CoTestingFilterAnalyzer
from amr_cascade_platform.cascade.analyzers.conditional_probability_analyzer import ConditionalProbabilityAnalyzer
from amr_cascade_platform.cascade.analyzers.escalation_ratio_analyzer import EscalationRatioAnalyzer
from amr_cascade_platform.cascade.analyzers.retained_edge_analyzer import RetainedEdgeAnalyzer
from amr_cascade_platform.cascade.statistics.downstream_testing_regression import DownstreamTestingRegression

if not drug_pairs.empty:
    filtered_pairs, cotesting_pairs = CoTestingFilterAnalyzer(settings).filter(drug_pairs)
    conditional_probabilities = ConditionalProbabilityAnalyzer(settings).analyze(filtered_pairs)
    escalation_results = EscalationRatioAnalyzer(settings).analyze(conditional_probabilities)
    empty_adjusted = DownstreamTestingRegression(settings, path_manager)._empty_results()
    retained_edges_pre_validation = RetainedEdgeAnalyzer(settings).analyze(escalation_results, empty_adjusted)

    print("drug_pair_episodes:", len(drug_pairs))
    print("filtered_pairs:", len(filtered_pairs))
    print("co-testing excluded rows/pairs table:", len(cotesting_pairs))
    print("conditional_probability rows:", len(conditional_probabilities))
    print("escalation_result rows:", len(escalation_results))
    print("retained pre-validation edges:", len(retained_edges_pre_validation))
    display(escalation_results.sort_values("escalation_ratio", ascending=False).head(10))
else:
    filtered_pairs = conditional_probabilities = escalation_results = retained_edges_pre_validation = pd.DataFrame()
    print("Gold pair table missing; run gold build first.")


## 7. Validation Results

Validation is the slow stage. This notebook reads the validation artifact if it exists. It does not rerun permutation/bootstrap by default.


In [ ]:
validation_results = read_parquet_if_exists(expected["cascade_validation_results"])
edge_report = read_parquet_if_exists(expected["cascade_edge_report"])

if not validation_results.empty:
    display(validation_results["validation_status"].value_counts(dropna=False).rename_axis("validation_status").reset_index(name="edge_n"))
    display(validation_results.head())
else:
    print("No validation_results.parquet found. Adjusted-model and prevalence checks below will be limited.")

if not edge_report.empty:
    display(edge_report["validation_status"].value_counts(dropna=False).rename_axis("validation_status").reset_index(name="edge_n"))
    cols = [c for c in ["upstream_antibiotic", "downstream_antibiotic", "cascade_direction", "escalation_ratio", "validation_status", "adjusted_odds_ratio", "supports_adjusted_model"] if c in edge_report.columns]
    display(edge_report.loc[:, cols].head(20))


## 8. Adjusted Downstream-Testing Regression Input Audit

This is the key corrected behavior.

The adjusted model should use only pairs with validation status `robust` or `supported`. It should not receive all support-passing edges.


In [ ]:
if not escalation_results.empty and not validation_results.empty:
    validated_edge_results = CascadeAnalysisWorkflow._validated_edge_results(escalation_results, validation_results)
    valid_keys = validated_edge_results[["upstream_antibiotic", "downstream_antibiotic"]].drop_duplicates()
    all_support_keys = escalation_results.loc[escalation_results["passes_support_threshold"].eq(True), ["upstream_antibiotic", "downstream_antibiotic"]].drop_duplicates()

    print("support-passing candidate edges:", len(all_support_keys))
    print("validated robust/supported edges entering adjusted model:", len(valid_keys))
    display(validated_edge_results.head(20))
else:
    validated_edge_results = pd.DataFrame()
    print("Cannot construct adjusted-model input because escalation or validation results are unavailable.")


In [ ]:
adjusted_diagnostics = read_parquet_if_exists(expected["cascade_adjusted_diagnostics"])

if not adjusted_diagnostics.empty and not validation_results.empty:
    modeled_keys = adjusted_diagnostics[["upstream_antibiotic", "downstream_antibiotic"]].drop_duplicates()
    expected_keys = CascadeAnalysisWorkflow._validated_edge_results(escalation_results, validation_results)[["upstream_antibiotic", "downstream_antibiotic"]].drop_duplicates() if not escalation_results.empty else pd.DataFrame(columns=["upstream_antibiotic", "downstream_antibiotic"])

    extra = modeled_keys.merge(expected_keys, on=["upstream_antibiotic", "downstream_antibiotic"], how="left", indicator=True).query("_merge == 'left_only'")
    missing = expected_keys.merge(modeled_keys, on=["upstream_antibiotic", "downstream_antibiotic"], how="left", indicator=True).query("_merge == 'left_only'")

    print("adjusted diagnostics rows:", len(adjusted_diagnostics))
    print("modeled pair keys:", len(modeled_keys))
    print("unexpected adjusted pairs not robust/supported:", len(extra))
    print("validated pairs missing adjusted diagnostic row:", len(missing))
    if "non_estimable_reason" in adjusted_diagnostics.columns:
        display(adjusted_diagnostics["non_estimable_reason"].fillna("estimable").value_counts().rename_axis("reason").reset_index(name="pair_n"))
    else:
        print("ISSUE: adjusted diagnostics artifact lacks non_estimable_reason; it was generated before the current model-audit schema.")
    if len(extra):
        print("ISSUE: adjusted diagnostics contain pairs outside robust/supported validation set; regenerate cascade outputs.")
        display(extra.head(20))
else:
    print("No adjusted diagnostics artifact yet, or validation missing. This is expected before the next cascade run.")


## 9. Covariates Entering the Adjusted Model

The primary adjusted model uses leakage-safe episode covariates. Lab/vital summaries are excluded from primary adjustment because the extracts do not prove they occurred before culture order.

Comorbidity is used in two forms:

- `cov_comorbidity_count`: active comorbidity burden at culture.
- `comorb_*`: frequent active component flags, included only when supported inside a given pair model.


In [ ]:
from amr_cascade_platform.cascade.statistics.cascade_covariate_builder import CascadeCovariateBuilder

if not culture_episodes.empty:
    covariates = CascadeCovariateBuilder(settings, path_manager).build(culture_episodes)
    print("covariate rows:", len(covariates), "columns:", len(covariates.columns))
    comorb_cols = sorted([c for c in covariates.columns if c.startswith("comorb_")])
    timing_limited = sorted([c for c in covariates.columns if c in DownstreamTestingRegression._TIMING_LIMITED_ACUITY_COVARIATES])
    print("comorb_* component columns:", len(comorb_cols))
    print("timing-limited lab/vital columns present but excluded from primary model:", len(timing_limited))
    display(pd.DataFrame({"comorbidity_component_columns": comorb_cols[:50]}))
    display(covariates.head())
else:
    covariates = pd.DataFrame()
    print("culture_episodes missing; cannot build covariates.")


In [ ]:
# Inspect which covariates would enter one adjusted pair model.
if not filtered_pairs.empty and not validated_edge_results.empty and not culture_episodes.empty:
    regression = DownstreamTestingRegression(settings, path_manager)
    first = validated_edge_results.iloc[0]
    pair_frame = filtered_pairs[
        (filtered_pairs["upstream_antibiotic"].eq(first["upstream_antibiotic"]))
        & (filtered_pairs["downstream_antibiotic"].eq(first["downstream_antibiotic"]))
    ].copy()
    pair_frame["upstream_positive"] = pair_frame["upstream_susceptibility"].map(regression._map_positive)
    pair_frame = pair_frame[pair_frame["upstream_positive"].isin([0, 1])]
    pair_frame = pair_frame.merge(covariates, on=join_keys, how="left", validate="many_to_one")
    covs_for_pair = regression._adjustment_covariates_for_frame(pair_frame)
    design = regression._build_design_matrix(pair_frame, include_upstream_positive=True)
    print("example pair:", first["upstream_antibiotic"], "->", first["downstream_antibiotic"])
    print("model rows:", len(pair_frame))
    print("covariates selected before dummy expansion:", len(covs_for_pair))
    print(covs_for_pair)
    print("design matrix shape:", None if design is None else design.shape)
    if design is not None:
        display(design.head())
else:
    print("Need filtered pairs, validated edges, and culture episodes to inspect pair-specific design matrix.")


## 10. Prevalence-Shift Input Audit

Correct production behavior:

- Use only downstream antibiotics from validated `robust` / `supported` **escalation** edges.
- Do not use upstream-only drugs, unrelated eligible drugs, or validated suppression edges as primary prevalence targets.


In [ ]:
from amr_cascade_platform.surveillance.prevalence_shift_analyzer import PrevalenceShiftAnalyzer

prevalence_analyzer = PrevalenceShiftAnalyzer(settings)
validated_downstream = prevalence_analyzer._validated_downstream_antibiotics(edge_report if not edge_report.empty else validation_results)
print("validated downstream escalation antibiotics for prevalence:", len(validated_downstream))
print(sorted(validated_downstream))

prevalence_shift = read_parquet_if_exists(expected["prevalence_shift"])
mnar_curve = read_parquet_if_exists(expected["prevalence_mnar_curve"])
delta_curve = read_parquet_if_exists(expected["prevalence_legacy_delta_curve"])

if not prevalence_shift.empty:
    reported_drugs = set(prevalence_shift["drug"].dropna().astype(str))
    extra_drugs = sorted(reported_drugs - set(validated_downstream))
    missing_drugs = sorted(set(validated_downstream) - reported_drugs)
    print("prevalence result drugs:", len(reported_drugs))
    print("drugs reported but not validated downstream escalation drugs:", extra_drugs)
    print("validated downstream drugs absent from prevalence result, usually due support thresholds:", missing_drugs)
    display(prevalence_shift.head(20))
else:
    print("No prevalence_shift.parquet yet. This is expected before rerunning prevalence after the code update.")


## 11. Interpreting Sensitivity Output Correctly

Do not require every sensitivity value to be below naive prevalence. That would be wrong.

Expected behavior:

- Lower bound is always `<= naive` because unknown binary outcomes are treated as susceptible.
- Upper bound is usually `>= naive` because unknown binary outcomes are treated as resistant.
- In MNAR odds-tilt sensitivity, positive `lambda` lowers expected resistance among unknown outcomes; negative `lambda` raises it.
- Therefore the positive-selection side should often be below naive, but the full curve intentionally spans assumptions above and below naive.


In [ ]:
if not prevalence_shift.empty:
    check = prevalence_shift.copy()
    check["lower_le_naive"] = check["prevalence_lower_bound"].le(check["naive_prevalence"])
    check["upper_ge_lower"] = check["prevalence_upper_bound"].ge(check["prevalence_lower_bound"])
    display(check[["organism", "drug", "naive_prevalence_pct", "prevalence_lower_bound_pct", "prevalence_upper_bound_pct", "mnar_lambda0_prevalence_pct", "standardised_prevalence_pct", "lower_le_naive", "upper_ge_lower"]].head(30))
    print("all lower bounds <= naive:", bool(check["lower_le_naive"].all()))
    print("all upper bounds >= lower bounds:", bool(check["upper_ge_lower"].all()))

if not mnar_curve.empty:
    display(mnar_curve.groupby("mnar_lambda").agg(
        drugs=("drug", "nunique"),
        mean_shift_from_naive_pct=("mnar_shift_from_naive_pct", "mean"),
        median_shift_from_naive_pct=("mnar_shift_from_naive_pct", "median"),
        estimated_rows=("mnar_status", lambda s: int((s == "estimated").sum())),
    ).reset_index())


## 12. Optional: Run Small Local Stages

This block is deliberately disabled. Only set `RUN_OPTIONAL_STAGES = True` when using a small test dataset or when you explicitly want the notebook to regenerate local artifacts.


In [ ]:
if RUN_OPTIONAL_STAGES:
    from amr_cascade_platform.data.gold.gold_build_manager import GoldBuildManager, GoldBuildRequest
    from amr_cascade_platform.data.pipelines.gold_pipeline import GoldPipeline
    from amr_cascade_platform.cascade.workflows.cascade_analysis_workflow import CascadeAnalysisRequest
    from amr_cascade_platform.cascade.workflows.cascade_analysis_workflow import CascadeAnalysisWorkflow
    from amr_cascade_platform.surveillance.workflows.prevalence_shift_workflow import PrevalenceShiftWorkflow, PrevalenceShiftRequest

    GoldPipeline(GoldBuildManager(settings, path_manager)).run(
        GoldBuildRequest(source_scope="combined", site=None, organism=ORGANISM_LABEL)
    )
    CascadeAnalysisWorkflow(settings, path_manager).run(
        CascadeAnalysisRequest(gold_scope="combined", site=None, organism=ORGANISM_SLUG)
    )
    PrevalenceShiftWorkflow(settings, path_manager).run(
        PrevalenceShiftRequest(scope="combined", site=None, organism=ORGANISM_SLUG)
    )
else:
    print("Optional local stage execution disabled. Set RUN_OPTIONAL_STAGES = True to run it.")


## 13. HPC Readiness Checklist

Before production HPC:

1. No old jobs should be running from the previous code contract.
2. `.venv` on HPC should be recreated cleanly, not reused from the broken Python 3.9/3.12 mixed environment.
3. Gold, cascade, validation merge, adjusted diagnostics, prevalence, and reports must be regenerated.
4. After production, rerun this notebook against the HPC-returned artifacts or copied `logs_outputs` to verify:
   - adjusted diagnostics pair keys are a subset of robust/supported validation keys;
   - prevalence drugs are a subset of validated downstream escalation drugs;
   - lower bounds are <= naive;
   - full MNAR/delta curves are interpreted as sensitivity ranges, not forced corrections.
